# Separate survey data into one long-format CSV

This notebook:

- reads every CSV in `input_dir`
- keeps **survey rows only**
- **excludes consent**
- writes **one output file only**: `output_survey/all_survey_rows.csv`
- puts **each survey question** and **each summary score** into **separate rows**
- drops columns that are **entirely empty across the final survey output**

It is also designed to handle malformed survey rows where standard `pandas.read_csv()` would skip or misalign them.


In [1]:
from pathlib import Path

# Change this to your folder that contains the task CSV files
input_dir = "."

# Output folder name
output_folder_name = "output_survey"


In [2]:
import csv
import pandas as pd
from pathlib import Path

SURVEY_TYPES = {
    "bis_bas_survey",
    "need_for_cognition_survey",
    "five_dcr_survey",
    "color_blind_survey_plate",
    "color_blind_survey_summary",
    "post_survey_builder",
    "post_survey_demographics",
    "post_survey",
    "consent",
}

def nz(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    s = str(v).strip()
    return None if s == "" or s.lower() == "nan" else s

def add_record(records, **kwargs):
    clean = {}
    for k, v in kwargs.items():
        if v is None:
            continue
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        clean[k] = v
    records.append(clean)

def row_to_dict_standard(header, row):
    row_adj = row[:len(header)] + [""] * max(0, len(header) - len(row))
    return dict(zip(header, row_adj))

def repair_split_json_field(row, header_len, json_idx):
    extra = len(row) - header_len
    if extra <= 0:
        return row
    end = json_idx + extra + 1
    repaired = row[:json_idx] + [",".join(row[json_idx:end])] + row[end:]
    return repaired if len(repaired) == header_len else row

def read_survey_rows_custom(csv_path):
    rows = []
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        # The task CSV uses backslash-escaped quotes inside JSON-like fields.
        # Without escapechar="\\", csv.reader splits those fields and shifts later columns.
        reader = csv.reader(f, escapechar="\\")
        header = next(reader)

        for line_no, row in enumerate(reader, start=2):
            if len(row) < 3:
                continue

            trial_type = row[2].strip()
            if trial_type not in SURVEY_TYPES or trial_type == "consent":
                continue

            # Safety fallback only if a row is still wider than the header even after
            # using escapechar="\\"
            if len(row) > len(header):
                if trial_type == "bis_bas_survey":
                    row = repair_split_json_field(row, len(header), json_idx=103)
                elif trial_type == "need_for_cognition_survey":
                    row = repair_split_json_field(row, len(header), json_idx=194)

            rec = row_to_dict_standard(header, row)
            rec["_source_line_no"] = line_no
            rows.append(rec)

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)

def canonical_survey_name(trial_type):
    mapping = {
        "bis_bas_survey": "bis_bas_survey",
        "need_for_cognition_survey": "need_for_cognition_survey",
        "five_dcr_survey": "five_dcr_survey",
        "color_blind_survey_plate": "color_blind_survey",
        "color_blind_survey_summary": "color_blind_survey",
        "post_survey_builder": "post_survey_builder",
        "post_survey_demographics": "post_survey_demographics",
        "post_survey": "post_survey",
    }
    return mapping.get(trial_type, trial_type)

def dedupe_keep_first(long_df):
    if long_df.empty:
        return long_df

    out = long_df.copy()

    # Put rows in earliest-occurrence order so drop_duplicates keeps the first real record.
    for col in ["trial_index", "source_row_line", "time_elapsed"]:
        if col in out.columns:
            out[f"__sort_{col}"] = pd.to_numeric(out[col], errors="coerce")

    sort_cols = [c for c in ["participant_id", "survey_name", "__sort_trial_index", "__sort_source_row_line", "__sort_time_elapsed"] if c in out.columns]
    if sort_cols:
        out = out.sort_values(sort_cols, kind="stable").reset_index(drop=True)

    # Treat repeated copies of the same participant/item/value block as duplicates.
    # Ignore timing/source-position columns so later re-logged copies are removed.
    dedupe_cols = [
        c for c in [
            "source_file",
            "participant_id",
            "survey_name",
            "source_trial_type",
            "record_type",
            "item_id",
            "item_label",
            "raw_value",
            "scored_value",
            "reverse_flag",
            "correct_value",
            "is_scored_item",
            "plate_index",
            "plate_id",
            "plate_label",
            "image_path",
            "color",
            "slider_stem_pct",
            "slider_cap_pct",
            "desired_stem",
            "desired_cap",
            "chosen_filename",
            "chosen_stem",
            "chosen_cap",
        ]
        if c in out.columns
    ]

    if dedupe_cols:
        out = out.drop_duplicates(subset=dedupe_cols, keep="first").reset_index(drop=True)

    helper_cols = [c for c in out.columns if c.startswith("__sort_")]
    if helper_cols:
        out = out.drop(columns=helper_cols)

    return out

def survey_rows_to_long(survey_df, source_file):
    if survey_df.empty or "trial_type" not in survey_df.columns:
        return pd.DataFrame()

    records = []
    file_has_demo = (survey_df["trial_type"] == "post_survey_demographics").any()

    for _, row in survey_df.iterrows():
        trial_type = nz(row.get("trial_type"))
        participant_id = nz(row.get("id"))

        base = {
            "source_file": source_file,
            "participant_id": participant_id,
            "survey_name": canonical_survey_name(trial_type),
            "source_trial_type": trial_type,
            "source_row_line": nz(row.get("_source_line_no")),
            "trial_index": nz(row.get("trial_index")),
            "rt": nz(row.get("rt")),
            "time_elapsed": nz(row.get("time_elapsed")),
        }

        if trial_type == "bis_bas_survey":
            for i in range(1, 21):
                raw = nz(row.get(f"bisbas_item_{i}_raw"))
                scored = nz(row.get(f"bisbas_item_{i}_scored"))
                reverse_flag = nz(row.get(f"bisbas_item_{i}_reverse"))
                if raw or scored or reverse_flag:
                    add_record(
                        records,
                        **base,
                        record_type="question",
                        item_id=f"item_{i}",
                        item_label=f"BIS/BAS item {i}",
                        raw_value=raw,
                        scored_value=scored,
                        reverse_flag=reverse_flag,
                    )

            for col in [
                "bisbas_raw_total",
                "bisbas_raw_mean",
                "bisbas_scored_total",
                "bisbas_scored_mean",
                "bis_score_sum",
                "bis_score_mean",
                "bas_drive_sum",
                "bas_drive_mean",
                "bas_fun_seeking_sum",
                "bas_fun_seeking_mean",
                "bas_reward_responsiveness_sum",
                "bas_reward_responsiveness_mean",
                "bas_total_sum",
                "bas_total_mean",
            ]:
                val = nz(row.get(col))
                if val:
                    add_record(
                        records,
                        **base,
                        record_type="score",
                        item_id=col,
                        item_label=col,
                        raw_value=val,
                    )

        elif trial_type == "need_for_cognition_survey":
            for i in range(1, 19):
                raw = nz(row.get(f"nfc_item_{i}_raw"))
                scored = nz(row.get(f"nfc_item_{i}_scored"))
                reverse_flag = nz(row.get(f"nfc_item_{i}_reverse"))
                if raw or scored or reverse_flag:
                    add_record(
                        records,
                        **base,
                        record_type="question",
                        item_id=f"item_{i}",
                        item_label=f"NFC item {i}",
                        raw_value=raw,
                        scored_value=scored,
                        reverse_flag=reverse_flag,
                    )

            for col in [
                "nfc_raw_total",
                "nfc_raw_mean",
                "nfc_scored_total",
                "nfc_scored_mean",
            ]:
                val = nz(row.get(col))
                if val:
                    add_record(
                        records,
                        **base,
                        record_type="score",
                        item_id=col,
                        item_label=col,
                        raw_value=val,
                    )

        elif trial_type == "five_dcr_survey":
            reverse_items = {9, 10, 11, 12}
            for i in range(1, 25):
                raw = nz(row.get(f"fiveDCR_q{i}"))
                scored = nz(row.get(f"fiveDCR_q{i}_reversed")) if i in reverse_items else raw
                if raw or scored:
                    add_record(
                        records,
                        **base,
                        record_type="question",
                        item_id=f"q{i}",
                        item_label=f"5DCR question {i}",
                        raw_value=raw,
                        scored_value=scored,
                        reverse_flag="1" if i in reverse_items else "0",
                    )

            for col in [
                "joyous_exploration_avg",
                "deprivation_sensitivity_avg",
                "stress_tolerance_avg",
                "thrill_seeking_avg",
                "general_social_curiosity_avg",
                "covert_social_curiosity_avg",
            ]:
                val = nz(row.get(col))
                if val:
                    add_record(
                        records,
                        **base,
                        record_type="score",
                        item_id=col,
                        item_label=col,
                        raw_value=val,
                    )

        elif trial_type == "color_blind_survey_plate":
            add_record(
                records,
                **base,
                record_type="question",
                item_id=nz(row.get("plate_id")) or f"plate_{nz(row.get('plate_index'))}",
                item_label=nz(row.get("plate_label")),
                raw_value=nz(row.get("response")),
                scored_value=nz(row.get("is_correct")),
                correct_value=nz(row.get("correct_response")),
                is_scored_item=nz(row.get("scored")),
                plate_index=nz(row.get("plate_index")),
                plate_id=nz(row.get("plate_id")),
                plate_label=nz(row.get("plate_label")),
                image_path=nz(row.get("image_path")),
            )

        elif trial_type == "color_blind_survey_summary":
            for col in [
                "n_items_total",
                "n_items_scored",
                "n_correct_scored",
                "accuracy_scored",
                "passed_all_scored",
            ]:
                val = nz(row.get(col))
                if val:
                    add_record(
                        records,
                        **base,
                        record_type="score",
                        item_id=col,
                        item_label=col,
                        raw_value=val,
                    )

        elif trial_type == "post_survey_builder":
            add_record(
                records,
                **base,
                record_type="question",
                item_id=nz(row.get("color")),
                item_label=f"Post-survey builder {nz(row.get('color'))}",
                color=nz(row.get("color")),
                slider_stem_pct=nz(row.get("slider_stem_pct")),
                slider_cap_pct=nz(row.get("slider_cap_pct")),
                desired_stem=nz(row.get("desired_stem")),
                desired_cap=nz(row.get("desired_cap")),
                chosen_filename=nz(row.get("chosen_filename")),
                chosen_stem=nz(row.get("chosen_stem")),
                chosen_cap=nz(row.get("chosen_cap")),
            )

        elif trial_type == "post_survey_demographics":
            for field in [
                "gender",
                "age",
                "game_frequency",
                "weekly_game_hours_past_month",
                "gamer_identity",
            ]:
                val = nz(row.get(field))
                if val:
                    add_record(
                        records,
                        **base,
                        record_type="question",
                        item_id=field,
                        item_label=field,
                        raw_value=val,
                    )

        elif trial_type == "post_survey":
            fields = [
                "enjoyment",
                "difficulty",
                "strategy",
                "color_rank_most_to_least",
                "room_rank_easiest_to_hardest",
                "builder_color_order",
            ]
            if not file_has_demo:
                fields = [
                    "gender",
                    "age",
                    "game_frequency",
                    "weekly_game_hours_past_month",
                    "gamer_identity",
                ] + fields

            for field in fields:
                val = nz(row.get(field))
                if val:
                    add_record(
                        records,
                        **base,
                        record_type="question",
                        item_id=field,
                        item_label=field,
                        raw_value=val,
                    )

    out = pd.DataFrame(records)
    if out.empty:
        return out

    out = dedupe_keep_first(out)
    out = out.dropna(axis=1, how="all")

    preferred = [
        "source_file",
        "participant_id",
        "survey_name",
        "source_trial_type",
        "record_type",
        "item_id",
        "item_label",
        "raw_value",
        "scored_value",
        "reverse_flag",
        "correct_value",
        "is_scored_item",
        "trial_index",
        "source_row_line",
        "rt",
        "time_elapsed",
        "plate_index",
        "plate_id",
        "plate_label",
        "image_path",
        "color",
        "slider_stem_pct",
        "slider_cap_pct",
        "desired_stem",
        "desired_cap",
        "chosen_filename",
        "chosen_stem",
        "chosen_cap",
    ]
    cols = [c for c in preferred if c in out.columns] + [c for c in out.columns if c not in preferred]
    out = out[cols]

    return out

def process_folder(input_dir, output_folder_name="output_survey"):
    input_dir = Path(input_dir)
    output_dir = input_dir / output_folder_name
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_files = [
        p for p in sorted(input_dir.glob("*.csv"))
        if not (
            p.name == "all_survey_rows.csv"
            or p.name.startswith("combined_")
            or p.parent.name.startswith(output_folder_name)
        )
    ]
    if not csv_files:
        raise FileNotFoundError(f"No raw task CSV files found in: {input_dir}")

    all_long = []
    for csv_file in csv_files:
        survey_df = read_survey_rows_custom(csv_file)
        if survey_df.empty:
            continue

        long_df = survey_rows_to_long(survey_df, csv_file.name)
        if not long_df.empty:
            all_long.append(long_df)

    if not all_long:
        raise ValueError("No survey rows were found in the CSV files.")

    final_df = pd.concat(all_long, ignore_index=True)
    final_df = final_df.dropna(axis=1, how="all")

    out_path = output_dir / "all_survey_rows.csv"
    final_df.to_csv(out_path, index=False)

    return final_df, out_path


In [3]:
final_df, out_path = process_folder(input_dir=input_dir, output_folder_name=output_folder_name)

print(f"Saved: {out_path}")
print(f"Shape: {final_df.shape}")

display(final_df.head(20))
display(final_df.groupby(["survey_name", "record_type"]).size().reset_index(name="n_rows"))


Saved: output_survey/all_survey_rows.csv
Shape: (5567, 28)


,source_file,participant_id,survey_name,source_trial_type,record_type,item_id,item_label,raw_value,scored_value,reverse_flag,...,plate_label,image_path,color,slider_stem_pct,slider_cap_pct,desired_stem,desired_cap,chosen_filename,chosen_stem,chosen_cap
0,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_1,BIS/BAS item 1,4,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_2,BIS/BAS item 2,3,3,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_3,BIS/BAS item 3,2,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_4,BIS/BAS item 4,4,4,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_5,BIS/BAS item 5,2,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_6,BIS/BAS item 6,2,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_7,BIS/BAS item 7,4,4,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_8,BIS/BAS item 8,3,3,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_9,BIS/BAS item 9,4,4,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,data_55bb9ae7fdf99b26d27fda01.csv,55bb9ae7fdf99b26d27fda01,bis_bas_survey,bis_bas_survey,question,item_10,BIS/BAS item 10,1,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,survey_name,record_type,n_rows
0,bis_bas_survey,question,980
1,bis_bas_survey,score,686
2,color_blind_survey,question,196
3,color_blind_survey,score,245
4,five_dcr_survey,question,1176
5,five_dcr_survey,score,294
6,need_for_cognition_survey,question,882
7,need_for_cognition_survey,score,196
8,post_survey,question,275
9,post_survey_builder,question,392
